## Sonifying Data using PLUM - A Case Study using the St Patrick's Day Storm

In our first sonification tutorial of SoRBET, we'll have a look at its most readily-usable function from PLUM (Python Libraries for Univariate Mappings), which take a single data source and output a single sound file. These are the most approachable as they require only 4 inputs to get cracking:

- The notes used in the musical score
- The length of the sound track you want to make
- The data being used to determine the sound property
- The data that represents the march of time in the data

Below, we'll see how 4 of PLUM's sonification functions work using the context of a **geomagnetic storm** - an event in Near-Earth space that upsets the state of this environment, often causing impacts on Earth (such as the aurora, or problems with power grids). In today's tutorial, we'll use some of the data measured by a NASA probe (one of the Van Allen Probes) to explore, using sound, where the probe was and what was happening in the plasma environment in the week that this storm took place.

______


## Setup and Imports

First, let's import everything we need. This should be familiar from previous sessions.

In [ ]:
%pip --quiet install strauss;
%pip --quiet install pydub

# Standard imports
import numpy as np
import matplotlib.pyplot as plt

#Point to relevant subroutines and data folders
import sys
sys.path.insert(0, '../subroutines/')
sys.path.insert(0, '../Data/')

# Import the PLUM object sonification routine
from PLUM import PitchSonification, PitchEventSonification, CutoffSonification, PanSonification
from ICE import note, chord, scale 
from SpORC import animate_l_mlt

# For audio processing, import routine from mixer
from Mixer import combine_audio

## The Van Allen Probe Data
In the block below, we will load the data in for this tutorial from 2 csv files - one continuously measured in time and one with "snapshots" of positions for event sonifications later. The data sets are as follows:

- `time_hr` (hrs): the time that has elapsed since midnight on March 14th
- `L` (number): The radial (i.e. distance from Earth measured along the equator) position of the probe, measured in multiples of the Earth's radius (so gives between 1-7 for this probe, typically, rather than huge kilometer ones!)
- `MLT` (number): The Magnetic Local Time (MLT) of the space craft, which determines the angular position of the spacecraft . The side of Earth furthest from the Sun is MLT = 0, the side closest to the sun is MLT = 12 and MLT increases anticlockwise when viewed from above the North Pole
- `N_e` (cm^-3): the number of electrons, on average, per cubic centimetre.
- `Bmag` (nanoteslas): The strength of Earth's magnetic field measured by the spacecraft, typically denoted as B or |B|.
- `Wavemags` (number): the strength of Chorus waves (an important electromagnetc wave in space weather) measured by the probe relative to the Earth's magnetic field.

These are loaded and plotted below. The vertical dashed grey line denotes the time at which the St Patrick's Day Storm started in the magnetosphere. See if you can spot any changes in any of the datasets after this point!

The other datasets follow the same principles as above, but are taken as snapshots of the moment the probe crosses a whole numbered L shell (e.g. L=1,2,3,4,5,6).

In [ ]:
time_hr, L, MLT, N_e, Bmag, Wavemags = np.loadtxt('../Data/patricks_day_storm_vanallenA_data.csv', delimiter=',', skiprows=1, unpack=True)

t_pos, L_pos, MLT_pos = np.loadtxt('../Data/patricks_day_storm_vanallenA_LCrossing_data.csv', delimiter=',', skiprows=1, unpack=True)

#Visualise it by creating a 4-by-1 set of axes
fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)

# Panel 1: radial position L
ax = axes[0]
ax.plot(time_hr/ 24, L, 'b-', linewidth=0.5)
ax.axvline(3+7/12, color='grey', linestyle='--', linewidth=1)
ax.set_ylabel('L shell')
ax.set_title('Van Allen Probe A data, 14-21th March 2015')

# Panel 2: azimuthal position MLT, in terms of magnetic hours
ax = axes[1]
ax.plot(time_hr/ 24, 0.5*(1+np.sin(np.unwrap(np.pi*MLT/12))), 'k-', linewidth=0.5)
ax.axvline(3+7/12, color='grey', linestyle='--', linewidth=1)
ax.set_ylabel('MLT (mag. hrs)')

# Panel 3: Number of electrons per cm cubed, N_e
ax = axes[2]
ax.plot(time_hr/ 24, N_e, 'r-', linewidth=0.5)
ax.axvline(3+7/12, color='grey', linestyle='--', linewidth=1)
ax.set_ylabel('n_e [cm⁻³]')

# Panel 4: strength of magnetic field |B|
ax = axes[3]
ax.plot(time_hr/ 24, Bmag, 'g-', linewidth=0.5)
ax.axvline(3+7/12, color='grey', linestyle='--', linewidth=1)
ax.set_ylabel('|B| [nT]')

# Panel 5: wave intensity relative to Earth's magnetic field |B_w|/|B|
ax = axes[4]
ax.plot(time_hr/ 24, Wavemags, 'm-', linewidth=0.5)
ax.axvline(3+7/12, color='grey', linestyle='--', linewidth=1)
ax.set_ylabel('|B_w|/|B|')
ax.set_xlabel('Days from 14th March 2015')

ax.set_xlim(0,7)

plt.tight_layout()
plt.show()

## Using PLUM to understand spacecraft positioning

In this first example, we're going to use some of the single function mappings to understand how the Van Allen probe was moving through space. To do this, we are going to sonify the L shell crossings (each time the spacecraft passes through a ring whose radius is a multiple of the Earth's) and its position around the Earth (often referred to as magnetic local time, MLT, ranging from 0-24). We use a **Pitch Event Sonification** for the L shells (to mark the "event") and a **pan sonification** for the MLT (to simulate the sound "moving")

What you should hear from each is
- You should hear the L shell sonification rise and fall in pitch, corresponding to the spacecraft travelling further away and returning closer to Earth. Listen out for the rate it does so. Are the notes evenly spaced in time, or does it get faster and slower? At which pitch (and thus distance from Earth) are the notes played fastest?
- You should hear the MLT sonification travel left and right across your speakers or headphones, indicating the spacecraft orbiting around the planet (and now through your head!). Is this orbit consistent, or does it have faster and slower portions?

In [ ]:
# Prescribe the notes for the event sonification - here, 5 are chosen across the A2 major scale as 
# the L_pos values have 5 categories, making the sound rich enough and harmonic
notes_L = scale("A2",'major',5)

# Set a series of notes for the pan sonification - here an A1 chord is chosen as it's 
# one octave lower than the L sonification, so as to harmonise with, but not interfere,
#  with the L shell sounds
notes_MLT = chord("A1",'major')

# Choose a common duration for the sonification. Don't make this too short or else a lot happens at once! 
length = 60

# Make the L sonification using PitchEventSonification
soni_L = PitchEventSonification(notes_L,length,t_pos,L_pos,time_lims=('0%','101%'),downsample=1)

print('Below is the sonification of when Probe A crosses each L shell, denoted by pitch of notes')

#Display the audo file below this command window
dobj_L = soni_L.notebook_display(show_waveform=False)
plt.close('all')

# Now make the MLT pan sonification using PanSonification - note the time limits have been input as 
# 101% in order to match the PitchEventSonification duration
# Interpolate to a finer uniform grid first

soni_MLT = PanSonification(notes_MLT,length,time_hr,0.5*(1+np.sin(np.unwrap(np.pi*MLT/12))),time_lims=('0%','101%'))
print('Below is the sonification of how Van Allen A moves around the planet')

#Display the audo file below this command window
dobj_MLT = soni_MLT.notebook_display(show_waveform=False)
plt.close('all')

How does this sound map up to what actually happens?

Now, currently these two sonifications occur separately, and so listening in tandem to get **a combined sense of position** is difficult if not impossible. One remedy would be to download the two wav files and open them in Audacity to combine them and listen to them together (which is valid!). Another way would be to fuse them in python - which can be done! An example of how this can be done using `pydub` is given below, and what it does is

- creates a .wav file for each of the above sonifications
- creates audiotracks from these .wav files using pydub's `AudioSegment`
- Overlays these two .wav files
- Save this overlaid file as a new, combined .wav file, here titled `VanAllen_Position.wav`
- Displays it as an audio player after the block

This then combines our two positional sonifications together into one single audio experience of the Van Allen probe's journey around space that week! This allows us to ask questions like

- Is the spacecraft orbiting fastest when it is furthest from Earth, or closest?
- Does it travel faster on the left or the right of Earth, or the same either side?

In [ ]:
combine_audio([soni_L,soni_MLT])

## Using PLUM to understand plasma environments
We can also use PLUM's functions to explore sonically the plasma data measured by the Van Allen probe. 

In this next example we'll explore the strength of Earth's magnetic field using a **Pitch Shift sonification** and the number of electrons measured by the probe using a **cutoff sonification**. 

We set a limit on the portion of the data explored using `time_lims` which is fed into the function. You may want to use the plot we made at the start to choose an exciting time to explore!

What you should hear in the following two sonifications is the following:
- In the magnetic field sonification, you should hear a low note occasionally rise to a higher pitch as the magnetic field gets stronger. Is there an equal amount of low and high pitch notes?
- In the number density sonification, you should find the sound "ramps up" and becomes richer as the probe encounters more electrons. Is the richness consistent throughout the piece or are there any places where the sound quality changes?

In [ ]:
# Prescribe the notes for the magnetic field pitch map sonification. We choose a bassier E1 note to make a nice rumble
notes_B = note("E1")

# Set a series of notes for the cutoff sonification - here an C3 chord is chosen
notes_ne = chord("C3")

#Set up a common portion of time. In our data, which has 7 days, one day's worth is about 14%.
#  To hear all days, put the end at 100%
#To hear specific days, make the starts and ends of your intervals different multiples of 14%
time_lims=('50%','66%')

# Choose a common duration for the sonifications. 
# For longer time durations above, consider making this longer, or a lot of things will happen all at once!
length = 60

# Make the magnetic field sonification using PitchSonification. We choose a 1 day limit for this as 15% of the data.
# Feel free to play with this limit and see what happens
soni_B = PitchSonification(notes_B,length,time_hr,Bmag,time_lims=time_lims)
print('Below is the sonification of the magnetic field observed by Van Allen Probe A')
dobj_B = soni_B.notebook_display(show_waveform=False)
plt.close('all')

# Now make the number density sonification using CutoffSonification 
soni_ne = CutoffSonification(notes_ne,length,time_hr,N_e,time_lims=time_lims)
print('Below is the sonification of the number of electrons encountered by Van Allen Probe A')
dobj_ne = soni_ne.notebook_display(show_waveform=False)
plt.close('all')

Taking this example further, this is a good time to point out that **we need not put just raw data into a sonification tool** and we can process it before sonifying it. This is useful in space physics where the data can span many orders of magnitude, and so only the largest parts of the data emerge in the sonifications.

below, we repeat the above sonification, but now define `Bdata` and `Ndata` as our input data for the magnetic and number density sonifications respective. On the first line of the function below, we may then tinker with the magnetic and number density data to make out sonifications more dynamic or to convey more information. Here, we've chosen logarithms to explore the order of magnitude of the data this time, rather than the exact values.

Compare these to the sonifications produced above - what's changed? Is there more, or less, going on in your new sonifications?

If you feel adventurous, change the way that `Bdata` and `Ndata` are defined by manipulating `Bmag` and `N_e` in different ways. Fun ones to try are

`Bdata = np.log(Wavemags)`

which provides a sense of wave activity in the magnetosphere, and

`Ndata = N_e/Bmag**2`

which is a special ratio in plasma physics that relates to the frequency at which plasma disturbances oscillate!

In [ ]:
#Define our modified/processed data here - we start with np.log to have a look at the 'size' of the data
Bdata = np.log(Bmag)
Ndata = np.log(N_e)

# Make the magnetic field sonification using PitchSonification. We choose a 1 day limit for this as 15% of the data.
# Feel free to play with this limit and see what happens

soni_B = PitchSonification(notes_B,length,time_hr,Bdata,time_lims=time_lims)

print('Below is the sonification of the magnetic field observed by Van Allen Probe A, with some tinkering')
dobj_B = soni_B.notebook_display(show_waveform=False)
plt.close('all')

# Now make the number density sonification using CutoffSonification 
soni_ne = CutoffSonification(notes_ne,length,time_hr,Ndata,time_lims=time_lims)
print('Below is the sonification of the number of electrons encountered by Van Allen Probe A, with some tinkering')
dobj_ne = soni_ne.notebook_display(show_waveform=False)
plt.close('all')

This completes our first tutorial on the basics of SoRBET, and hopefully gives you a straightforward appreciation of how spacecraft data can be turned into sound! All of the sonifications in SoRBET use the same basic approach - define the data, choose how this will alter the properties of sound, prescribe notes, choose a duration, and you're all set to go!

This tutorial should give you the basics of how functions in the `PLUM` set work, and hopefully introduce you to several ways in which data can be mapped to sound. In later tutorials, we will use multiple mappings in our sonifications, each controlling a different property of the sound, to generate richer sonifications to put the data into different relationships to better our understanding of the data.